# MAI511-2 Advanced Machine Learning — Regularized Regression

## Regularized Regression using Lasso and Ridge with Cross-Validation and Grid Search

**Objective:** Implement and compare Linear Regression, Lasso Regression and Ridge Regression. Use **5-fold GridSearchCV** to identify the optimal regularization parameter **alpha** for Lasso and Ridge.

### Dataset
`student_burnout.csv` — 2,000 records and 17 attributes.

**Target:** `burnout_score` (continuous numeric score)

### Required lab flow
1. Dataset loading and identification of features/target
2. Exploration: shape, missing values, duplicates, descriptive statistics, relationships
3. Preprocessing: missing-value handling, categorical encoding, train/test split, scaling
4. Baseline Linear Regression
5. Lasso + GridSearchCV
6. Ridge + GridSearchCV
7. Model comparison + coefficient comparison
8. Actual-vs-predicted, coefficients, CV/alpha plots
9. Interpretation of alpha, coefficients and scaling
10. Self-learning extension focused on reducing error

> **Important:** `high_burnout` is excluded because it is derived from the target and would cause target leakage. `student_id` is an identifier rather than a predictive feature.


In [ ]:
# Import the libraries used throughout the experiment.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the exact dataset supplied for this lab.
df = pd.read_csv("student_burnout.csv")

df.head()


## 1. Dataset exploration

The dataset contains student-related numerical variables plus one categorical variable (`gender`).

The target is `burnout_score`. Although the CSV also contains `high_burnout`, that column is not used as an input because it is derived from burnout and would leak target information into the model.


In [ ]:
# Dimensions: rows x columns
print("Dataset shape:", df.shape)

# Data types help us identify numeric and categorical variables.
print("\nData types:")
print(df.dtypes)

# Missing-value check.
print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False))

# Duplicate records.
print("\nDuplicate rows:", df.duplicated().sum())

# Descriptive statistics.
print("\nDescriptive statistics:")
display(df.describe(include="all").T)


In [ ]:
# Visualize the target distribution.
plt.figure(figsize=(8, 4))
sns.histplot(df["burnout_score"], discrete=True)
plt.title("Distribution of burnout_score")
plt.xlabel("Burnout score")
plt.show()

# Look at a few selected numeric features against the target.
selected_features = ["sleep_hours", "homework_hours", "screen_time_hours", "self_rated_stress"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feature in zip(axes.flat, selected_features):
    sns.regplot(
        data=df, x=feature, y="burnout_score",
        scatter_kws={"alpha": 0.35},
        line_kws={"linewidth": 2},
        ax=ax
    )
    ax.set_title(f"{feature} vs burnout_score")
plt.tight_layout()
plt.show()


## 2. Define X and y

- `X` contains independent variables.
- `y` contains the continuous target.
- `student_id` is removed because it is only an identifier.
- `high_burnout` is removed because it is derived from the target.


In [ ]:
X = df.drop(columns=["burnout_score", "high_burnout", "student_id"])
y = df["burnout_score"]

categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object", "category"]).columns.tolist()

print("Numeric features:", numeric_cols)
print("Categorical features:", categorical_cols)
print("Number of predictors:", X.shape[1])


## 3. Preprocessing

### Why scaling matters
Lasso and Ridge add a penalty based on coefficient size. If one feature is measured in very large units while another is measured in small units, the penalty can affect them unfairly. StandardScaler places numeric variables on a comparable scale.

### Why use a Pipeline?
The imputer, encoder and scaler are fitted as part of the training process. This keeps preprocessing consistent and reduces data leakage.

### Missing values
The supplied dataset has missing values in some numeric columns. Median imputation is used for numeric variables. `gender` is categorical, so missing categorical values would be filled using the most frequent category.


In [ ]:
# Build a reusable preprocessing transformer.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", drop="first"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

# 80/20 split. random_state=42 makes the experiment reproducible.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)


## 4. Baseline Linear Regression

This is the required baseline. It does not use a regularization penalty.

We evaluate it using:
- **MAE:** average absolute prediction error
- **MSE:** average squared error; larger errors are penalized more
- **RMSE:** square root of MSE; expressed in the target's units
- **R²:** proportion of variance explained by the model


In [ ]:
def evaluate_model(model, X_test, y_test):
    predictions = model.predict(X_test)
    return {
        "MAE": mean_absolute_error(y_test, predictions),
        "MSE": mean_squared_error(y_test, predictions),
        "RMSE": mean_squared_error(y_test, predictions) ** 0.5,
        "R²": r2_score(y_test, predictions)
    }

linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)
linear_metrics = evaluate_model(linear_model, X_test, y_test)

print(linear_metrics)


## 5. Lasso Regression + 5-fold Grid Search

**Lasso = L1 regularization.**

Its objective adds a penalty proportional to the absolute value of coefficients:

`Loss + alpha × sum(|coefficient|)`

A useful property of Lasso is that it can shrink some coefficients exactly to zero, which can act as a form of feature selection.

`GridSearchCV` tries multiple alpha values using **5-fold cross-validation**. We select the alpha with the best cross-validated RMSE.


In [ ]:
alpha_grid = np.logspace(-4, 2, 30)

lasso_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Lasso(max_iter=20000))
])

lasso_grid = GridSearchCV(
    lasso_pipeline,
    param_grid={"model__alpha": alpha_grid},
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

lasso_grid.fit(X_train, y_train)

lasso_metrics = evaluate_model(lasso_grid.best_estimator_, X_test, y_test)

print("Best Lasso alpha:", lasso_grid.best_params_["model__alpha"])
print("Lasso test metrics:", lasso_metrics)


## 6. Ridge Regression + 5-fold Grid Search

**Ridge = L2 regularization.**

Its objective adds a penalty proportional to the squared coefficients:

`Loss + alpha × sum(coefficient²)`

Ridge usually keeps coefficients non-zero but shrinks them. It is useful when predictors contain correlated information.

Again, 5-fold GridSearchCV chooses the alpha with the best cross-validated RMSE.


In [ ]:
ridge_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge())
])

ridge_grid = GridSearchCV(
    ridge_pipeline,
    param_grid={"model__alpha": alpha_grid},
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

ridge_grid.fit(X_train, y_train)

ridge_metrics = evaluate_model(ridge_grid.best_estimator_, X_test, y_test)

print("Best Ridge alpha:", ridge_grid.best_params_["model__alpha"])
print("Ridge test metrics:", ridge_metrics)


## 7. Model comparison

The same held-out 20% test set is used for all three required models so the comparison is fair.


In [ ]:
results = pd.DataFrame({
    "Linear Regression": linear_metrics,
    "Lasso": lasso_metrics,
    "Ridge": ridge_metrics
}).T

results


## 8. Actual vs predicted values

A good regression plot should have points reasonably close to the 45° reference line. This visual complements the numeric metrics.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, name, model in [
    (axes[0], "Lasso", lasso_grid.best_estimator_),
    (axes[1], "Ridge", ridge_grid.best_estimator_)
]:
    pred = model.predict(X_test)
    ax.scatter(y_test, pred, alpha=0.55)
    lo = min(y_test.min(), pred.min())
    hi = max(y_test.max(), pred.max())
    ax.plot([lo, hi], [lo, hi], linestyle="--")
    ax.set_xlabel("Actual burnout score")
    ax.set_ylabel("Predicted burnout score")
    ax.set_title(f"Actual vs Predicted — {name}")

plt.tight_layout()
plt.show()


## 9. Coefficient comparison

Because the preprocessing pipeline creates encoded and scaled features, we first recover the transformed feature names. Then we compare the coefficients from Linear Regression, Lasso and Ridge.

For Lasso, coefficients that become exactly (or effectively) zero indicate features that the L1 penalty removed from the fitted linear relationship.


In [ ]:
def coefficient_series(fitted_pipeline):
    prep = fitted_pipeline.named_steps["preprocessor"]
    model = fitted_pipeline.named_steps["model"]
    names = prep.get_feature_names_out()
    return pd.Series(model.coef_, index=names)

coef_comparison = pd.DataFrame({
    "Linear Regression": coefficient_series(linear_model),
    "Lasso": coefficient_series(lasso_grid.best_estimator_),
    "Ridge": coefficient_series(ridge_grid.best_estimator_)
}).fillna(0)

display(coef_comparison.sort_values("Lasso", key=np.abs, ascending=False).head(20))

zero_lasso = coef_comparison.index[np.isclose(coef_comparison["Lasso"], 0, atol=1e-8)]
print("Lasso coefficients that are effectively zero:")
print(list(zero_lasso))


## 10. Cross-validation performance against alpha

This directly demonstrates the GridSearchCV process. Lower cross-validated RMSE is preferred.


In [ ]:
lasso_cv = pd.DataFrame({
    "alpha": lasso_grid.cv_results_["param_model__alpha"].astype(float),
    "RMSE": -lasso_grid.cv_results_["mean_test_score"]
}).sort_values("alpha")

ridge_cv = pd.DataFrame({
    "alpha": ridge_grid.cv_results_["param_model__alpha"].astype(float),
    "RMSE": -ridge_grid.cv_results_["mean_test_score"]
}).sort_values("alpha")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(lasso_cv["alpha"], lasso_cv["RMSE"], marker="o", markersize=3)
axes[0].axvline(float(lasso_grid.best_params_["model__alpha"]), linestyle="--")
axes[0].set_xscale("log")
axes[0].set_title("Lasso: 5-fold CV RMSE vs alpha")
axes[0].set_xlabel("alpha")
axes[0].set_ylabel("CV RMSE")

axes[1].plot(ridge_cv["alpha"], ridge_cv["RMSE"], marker="o", markersize=3)
axes[1].axvline(float(ridge_grid.best_params_["model__alpha"]), linestyle="--")
axes[1].set_xscale("log")
axes[1].set_title("Ridge: 5-fold CV RMSE vs alpha")
axes[1].set_xlabel("alpha")
axes[1].set_ylabel("CV RMSE")

plt.tight_layout()
plt.show()


# 11. Self-learning / additional exploration: Polynomial Ridge

To investigate how to reduce error further, we add pairwise interaction features with `PolynomialFeatures(degree=2, interaction_only=True)` and then use Ridge regularization.

Why this makes sense:
- Some effects may depend on combinations of variables rather than one variable alone.
- Interaction expansion increases the number of predictors.
- Ridge can control the resulting coefficient sizes.

This is an **extension**, not a replacement for the required Linear/Lasso/Ridge comparison.


In [ ]:
poly_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("poly", PolynomialFeatures(
            degree=2, include_bias=False, interaction_only=True
        ))
    ]), numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

poly_ridge_grid = GridSearchCV(
    Pipeline([
        ("preprocessor", poly_preprocessor),
        ("model", Ridge())
    ]),
    param_grid={"model__alpha": np.logspace(-2, 3, 25)},
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

poly_ridge_grid.fit(X_train, y_train)
poly_metrics = evaluate_model(poly_ridge_grid.best_estimator_, X_test, y_test)

print("Best Polynomial Ridge alpha:", poly_ridge_grid.best_params_["model__alpha"])
print("Polynomial Ridge metrics:", poly_metrics)


## 12. Final interpretation

### Effect of alpha
- Very small alpha means weak regularization and behaviour close to ordinary Linear Regression.
- Larger alpha means stronger shrinkage.
- Too much regularization can cause underfitting.

### Effect on coefficients
- **Lasso (L1):** can force coefficients to exactly zero → sparse model / feature selection.
- **Ridge (L2):** shrinks coefficients toward zero but usually keeps them non-zero.

### Why scaling is important
The regularization penalty acts on coefficient magnitude. Without scaling, variables measured on different scales can be penalized unevenly.

### Model suitability
Use the experimental test metrics and coefficient behavior rather than assuming one method is always better. The best choice for this dataset is the model that gives the most useful error/generalization trade-off while remaining interpretable.

### Self-learning conclusion
Polynomial Ridge tests whether pairwise interactions can reduce error. Its result should be reported honestly: an improvement on one fixed split is evidence for that split, not proof of universal superiority.
